# 06 — GroupBy & Aggregation
**Goal:** Summarize data by groups — the most important operation in analytics.

This is the pandas equivalent of SQL's `GROUP BY`. It answers questions like:
- What's the CVR per channel per month?
- Which week had the most activations?
- What's the average OTP drop-off by day of week?

Topics:
- `groupby()` + `agg()` — basic aggregations
- Named aggregations — clean, explicit syntax
- Multiple grouping keys
- `transform()` — add group stats back to original rows
- `apply(lambda)` vs `agg()` — when each one is needed
- Aggregaciones avanzadas — equivalente a `CASE WHEN` en SQL
- `pivot_table()` — matriz 2D
- `crosstab()` — tabla de frecuencias

In [1]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/funnel_data.csv', parse_dates=['date'])
df = df.assign(
    cvr     = lambda x: x['activacion_tarjeta'] / x['visita_landing'] * 100,
    month   = lambda x: x['date'].dt.month,
    weekday = lambda x: x['date'].dt.day_name(),
    week    = lambda x: x['date'].dt.isocalendar().week.astype(int),
)

print(df.shape)
df.head(3)

(450, 16)


,date,channel,visita_landing,inicio_solicitud,datos_personales,otp,datos_financieros,carga_documentos,evaluacion_crediticia,aprobacion,firma_digital,activacion_tarjeta,cvr,month,weekday,week
0,2024-01-01,organic,1010,388,291,207,165,99,89,48,39,27,2.673267,1,Monday,1
1,2024-01-01,paid,1021,321,240,163,130,78,70,32,26,18,1.762977,1,Monday,1
2,2024-01-01,email,481,202,151,102,89,53,47,25,20,14,2.910603,1,Monday,1


## 1. Basic `groupby` + aggregation

In [2]:
# Total activations per channel
df.groupby('channel')['activacion_tarjeta'].sum()

channel
direct      510
email      1226
organic    2667
paid       1468
social      466
Name: activacion_tarjeta, dtype: int64

In [3]:
# Multiple aggregations on one column
df.groupby('channel')['cvr'].agg(['mean', 'median', 'std', 'min', 'max']).round(2)

,mean,median,std,min,max
channel,,,,,
direct,2.28,2.29,0.17,1.88,2.61
email,2.87,2.88,0.09,2.67,3.07
organic,2.68,2.67,0.05,2.53,2.78
paid,1.70,1.71,0.05,1.56,1.79
social,1.35,1.36,0.14,1.09,1.56


## 2. Named aggregations — clean and explicit

In [4]:
# Named aggregations: result_column_name=('source_column', 'function')
# This is the cleanest pattern for analytics summaries
channel_summary = df.groupby('channel').agg(
    total_visits      = ('visita_landing',     'sum'),
    total_activations = ('activacion_tarjeta',  'sum'),
    avg_cvr           = ('cvr',                'mean'),
    median_cvr        = ('cvr',                'median'),
    std_cvr           = ('cvr',                'std'),
    n_days            = ('date',               'nunique'),
).round(2).reset_index()

# Compute overall CVR from totals (more accurate than mean of daily CVRs)
channel_summary['true_cvr'] = (
    channel_summary['total_activations'] / channel_summary['total_visits'] * 100
).round(2)

channel_summary.sort_values('true_cvr', ascending=False)

,channel,total_visits,total_activations,avg_cvr,median_cvr,std_cvr,n_days,true_cvr
1,email,42563,1226,2.87,2.88,0.09,90,2.88
2,organic,99551,2667,2.68,2.67,0.05,90,2.68
0,direct,22224,510,2.28,2.29,0.17,90,2.29
3,paid,86267,1468,1.70,1.71,0.05,90,1.70
4,social,34169,466,1.35,1.36,0.14,90,1.36


## 3. Multiple grouping keys

In [5]:
# CVR by channel AND month
monthly_channel = df.groupby(['month', 'channel']).agg(
    visits      = ('visita_landing',    'sum'),
    activations = ('activacion_tarjeta', 'sum'),
).reset_index()

monthly_channel['cvr'] = (monthly_channel['activations'] / monthly_channel['visits'] * 100).round(2)

print(monthly_channel.to_string())

    month  channel  visits  activations   cvr
0       1   direct    7306          165  2.26
1       1    email   13749          392  2.85
2       1  organic   32433          866  2.67
3       1     paid   27422          464  1.69
4       1   social   10915          141  1.29
5       2   direct    7164          164  2.29
6       2    email   14090          406  2.88
7       2  organic   32032          859  2.68
8       2     paid   27995          474  1.69
9       2   social   11078          153  1.38
10      3   direct    7754          181  2.33
11      3    email   14724          428  2.91
12      3  organic   35086          942  2.68
13      3     paid   30850          530  1.72
14      3   social   12176          172  1.41


In [6]:
# CVR by day of week — are weekends worse?
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

dow = df.groupby('weekday').agg(
    avg_cvr     = ('cvr', 'mean'),
    avg_visits  = ('visita_landing', 'mean'),
).round(2).reindex(weekday_order)  # reindex to correct order

print(dow)

           avg_cvr  avg_visits
weekday                       
Monday        2.19      662.83
Tuesday       2.20      646.31
Wednesday     2.19      667.00
Thursday      2.19      669.00
Friday        2.20      656.55
Saturday      2.13      567.55
Sunday        2.13      554.55


## 4. `transform()` — add group stats back to the original DataFrame
Unlike `agg()` which collapses rows, `transform()` returns a column with the same length as the original.

In [7]:
# Add each channel's mean CVR as a new column
# Useful for computing "how far is today from the channel average"
df['channel_avg_cvr'] = df.groupby('channel')['cvr'].transform('mean')
df['cvr_vs_avg']      = df['cvr'] - df['channel_avg_cvr']

print(df[['date','channel','cvr','channel_avg_cvr','cvr_vs_avg']].head(10).round(2))

        date  channel   cvr  channel_avg_cvr  cvr_vs_avg
0 2024-01-01  organic  2.67             2.68       -0.00
1 2024-01-01     paid  1.76             1.70        0.06
2 2024-01-01    email  2.91             2.87        0.04
3 2024-01-01   social  1.35             1.35       -0.00
4 2024-01-01   direct  2.33             2.28        0.04
5 2024-01-02  organic  2.65             2.68       -0.03
6 2024-01-02     paid  1.79             1.70        0.09
7 2024-01-02    email  2.80             2.87       -0.07
8 2024-01-02   social  1.35             1.35       -0.00
9 2024-01-02   direct  2.35             2.28        0.07


/var/folders/3m/z0qp5nwj1vlccby390hqt0kc0000gn/T/ipykernel_10273/3777517791.py:6: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(df[['date','channel','cvr','channel_avg_cvr','cvr_vs_avg']].head(10).round(2))


In [8]:
# Rank within group — which day was best for each channel?
df['daily_rank_in_channel'] = df.groupby('channel')['cvr'].rank(ascending=False).astype(int)

# Best day per channel
best_days = df[df['daily_rank_in_channel'] == 1][['channel','date','cvr']].sort_values('cvr', ascending=False)
print(best_days)

     channel       date       cvr
367    email 2024-03-14  3.071672
265  organic 2024-02-23  2.782194
389   direct 2024-03-18  2.614379
396     paid 2024-03-20  1.793339
323   social 2024-03-05  1.562500


## 5. `apply(lambda)` vs `agg()` — cuándo usar cada uno

`agg()` aplica una función **por columna** — cada función recibe una Serie.  
`apply(lambda)` recibe el **sub-DataFrame completo** del grupo, lo que te permite cruzar columnas dentro del cálculo.

```
groupby + agg()               groupby + apply(lambda x: ...)
─────────────────────         ──────────────────────────────
Cada función ve 1 columna     La lambda ve todas las columnas
Más rápido (vectorizado)      Más flexible (pero más lento)
Perfecto para sum/mean/std    Necesario cuando el cálculo
de columnas independientes    cruza columnas o tiene lógica
                              condicional compleja
```

**Regla práctica:** usa `agg()` siempre que puedas. Sólo cambia a `apply()` cuando necesites cruzar columnas dentro del mismo grupo.

In [9]:
# ── agg(): cada función opera sobre una sola columna ─────────────────────────
result_agg = df.groupby('channel').agg(
    total_visits      = ('visita_landing',    'sum'),
    total_activations = ('activacion_tarjeta', 'sum'),
    avg_cvr           = ('cvr',               'mean'),
).round(2)
print("Con agg():")
print(result_agg)

print()

# ── apply(): la lambda recibe el sub-DataFrame completo del grupo ─────────────
# CVR real = sum(activaciones) / sum(visitas) — requiere cruzar dos columnas
# No es posible con agg() porque cada función sólo ve una columna
result_apply = df.groupby('channel').apply(
    lambda x: pd.Series({
        'total_visits':      x['visita_landing'].sum(),
        'total_activations': x['activacion_tarjeta'].sum(),
        'true_cvr':          x['activacion_tarjeta'].sum() / x['visita_landing'].sum() * 100,
    })
).round(2)
print("Con apply(lambda) — CVR calculado desde los totales del grupo:")
print(result_apply)

Con agg():
         total_visits  total_activations  avg_cvr
channel                                          
direct          22224                510     2.28
email           42563               1226     2.87
organic         99551               2667     2.68
paid            86267               1468     1.70
social          34169                466     1.35

Con apply(lambda) — CVR calculado desde los totales del grupo:
         total_visits  total_activations  true_cvr
channel                                           
direct        22224.0              510.0      2.29
email         42563.0             1226.0      2.88
organic       99551.0             2667.0      2.68
paid          86267.0             1468.0      1.70
social        34169.0              466.0      1.36


## 6. Agregaciones avanzadas — equivalente a `CASE WHEN` dentro de `SUM()` en SQL

En SQL es común hacer esto:

```sql
SELECT
    channel,
    SUM(CASE WHEN cvr >= 2.5 THEN visita_landing ELSE 0 END) AS visits_high_cvr,
    COUNT(CASE WHEN cvr >= 2.5 THEN 1 END)                   AS days_high_cvr,
    SUM(activacion_tarjeta) FILTER (WHERE month = 1)         AS activations_jan
FROM funnel
GROUP BY channel
```

En pandas hay tres patrones según la complejidad del cálculo:

In [10]:
# ── Patrón 1: lambda dentro de agg() — condición sobre UNA columna ───────────
# La Serie que recibe la lambda es la columna entera del grupo.
# Equivale a: COUNT/SUM(CASE WHEN col OP valor THEN ... END)

result1 = df.groupby('channel').agg(
    total_days    = ('cvr', 'count'),
    days_high_cvr = ('cvr', lambda s: (s >= 2.5).sum()),        # COUNT(CASE WHEN cvr>=2.5)
    days_low_cvr  = ('cvr', lambda s: (s < 1.5).sum()),         # COUNT(CASE WHEN cvr<1.5)
    pct_high_cvr  = ('cvr', lambda s: (s >= 2.5).mean() * 100), # % días con CVR alto
)
print("Patrón 1 — lambda en agg() (condición sobre una columna):")
print(result1.round(1))

Patrón 1 — lambda en agg() (condición sobre una columna):
         total_days  days_high_cvr  days_low_cvr  pct_high_cvr
channel                                                       
direct           90             10             0          11.1
email            90             90             0         100.0
organic          90             90             0         100.0
paid             90              0             0           0.0
social           90              0            73           0.0


In [11]:
# ── Patrón 2: apply(lambda) — condición que CRUZA varias columnas ────────────
# agg() no puede hacer esto porque cada función sólo ve una columna.
# x es el sub-DataFrame del grupo completo.

result2 = df.groupby('channel').apply(lambda x: pd.Series({
    # SUM(CASE WHEN cvr >= 2.5 THEN visita_landing ELSE 0 END)
    'visits_on_high_cvr_days': x.loc[x['cvr'] >= 2.5, 'visita_landing'].sum(),

    # SUM(activacion_tarjeta) FILTER (WHERE month = 1)
    'activations_jan':         x.loc[x['month'] == 1, 'activacion_tarjeta'].sum(),

    # CVR calculado sólo en días de fin de semana
    'cvr_weekend': (
        x.loc[x['weekday'].isin(['Saturday', 'Sunday']), 'activacion_tarjeta'].sum() /
        x.loc[x['weekday'].isin(['Saturday', 'Sunday']), 'visita_landing'].sum() * 100
    ),
})).round(2)

print("Patrón 2 — apply(lambda) (condición que cruza columnas):")
print(result2)

Patrón 2 — apply(lambda) (condición que cruza columnas):
         visits_on_high_cvr_days  activations_jan  cvr_weekend
channel                                                       
direct                    2894.0            165.0         2.23
email                    42563.0            392.0         2.83
organic                  99551.0            866.0         2.66
paid                         0.0            464.0         1.67
social                       0.0            141.0         1.30


In [12]:
# ── Patrón 3: def — lógica compleja con múltiples ramas o pasos ──────────────
# Cuando hay más de 2 condiciones o variables intermedias, un def es más legible.
# Equivale a CASE WHEN anidados + lógica de negocio dentro de la agregación.

def channel_performance(x):
    total_visits      = x['visita_landing'].sum()
    total_activations = x['activacion_tarjeta'].sum()
    cvr               = total_activations / total_visits * 100

    # CASE WHEN cvr >= 2.5 THEN 'High' WHEN cvr >= 1.8 THEN 'Medium' ELSE 'Low' END
    if   cvr >= 2.5: tier = 'High'
    elif cvr >= 1.8: tier = 'Medium'
    else:            tier = 'Low'

    # Tendencia: CVR de marzo vs enero
    cvr_jan = (x.loc[x['month']==1, 'activacion_tarjeta'].sum() /
               x.loc[x['month']==1, 'visita_landing'].sum() * 100)
    cvr_mar = (x.loc[x['month']==3, 'activacion_tarjeta'].sum() /
               x.loc[x['month']==3, 'visita_landing'].sum() * 100)

    return pd.Series({
        'cvr':              round(cvr, 2),
        'tier':             tier,
        'trend_jan_to_mar': 'up' if cvr_mar > cvr_jan else 'down',
        'best_month':       x.groupby('month')['cvr'].mean().idxmax(),
    })

result3 = df.groupby('channel').apply(channel_performance)
print("Patrón 3 — def con lógica compleja:")
print(result3)

Patrón 3 — def con lógica compleja:
          cvr    tier trend_jan_to_mar  best_month
channel                                           
direct   2.29  Medium               up           3
email    2.88    High               up           3
organic  2.68    High               up           3
paid     1.70     Low               up           3
social   1.36     Low               up           3


In [13]:
# CVR by month (rows) × channel (columns)
pivot = df.pivot_table(
    index='month',           # rows
    columns='channel',       # columns
    values='cvr',            # values to aggregate
    aggfunc='mean',          # how to aggregate
).round(2)

print(pivot)

channel  direct  email  organic  paid  social
month                                        
1          2.25   2.85     2.67  1.69    1.28
2          2.27   2.88     2.68  1.69    1.38
3          2.32   2.90     2.68  1.71    1.40


In [14]:
# Add row and column totals with margins=True
pivot_with_totals = df.pivot_table(
    index='month',
    columns='channel',
    values='activacion_tarjeta',
    aggfunc='sum',
    margins=True,           # adds 'All' row and column
    margins_name='Total',
)

print(pivot_with_totals)

channel  direct  email  organic  paid  social  Total
month                                               
1           165    392      866   464     141   2028
2           164    406      859   474     153   2056
3           181    428      942   530     172   2253
Total       510   1226     2667  1468     466   6337


## 8. `crosstab()` — frequency table between two categorical columns

In [15]:
# How many days did each channel appear in each month?
ct = pd.crosstab(
    df['month'],
    df['channel'],
    margins=True,
    margins_name='Total'
)
print(ct)

channel  direct  email  organic  paid  social  Total
month                                               
1            31     31       31    31      31    155
2            29     29       29    29      29    145
3            30     30       30    30      30    150
Total        90     90       90    90      90    450


In [16]:
# Normalize to percentages
ct_pct = pd.crosstab(
    df['month'],
    df['channel'],
    normalize='index'   # normalize by row (each row sums to 1)
).round(2) * 100

print(ct_pct)

channel  direct  email  organic  paid  social
month                                        
1          20.0   20.0     20.0  20.0    20.0
2          20.0   20.0     20.0  20.0    20.0
3          20.0   20.0     20.0  20.0    20.0


## Summary

| Tool | Use when |
|---|---|
| `groupby('col').agg(...)` | Summarize by one grouping key |
| `groupby(['a','b']).agg(...)` | Summarize by multiple keys |
| Named agg: `result=('col','func')` | Cleanest pattern for reports |
| `.transform('mean')` | Add group stat back to original rows |
| `.rank()` | Rank within group |
| `agg(col=('col', lambda s: ...))` | `CASE WHEN` sobre **una** columna |
| `apply(lambda x: pd.Series({...}))` | `CASE WHEN` que **cruza** varias columnas |
| `apply(def_func)` | Lógica compleja con múltiples ramas o pasos |
| `pivot_table()` | 2D summary: rows × columns |
| `crosstab()` | Frequency count between two categories |
| `margins=True` | Add totals row/column |

### Equivalencias SQL → pandas
| SQL | pandas |
|---|---|
| `COUNT(CASE WHEN x > n THEN 1 END)` | `agg(c=('col', lambda s: (s > n).sum()))` |
| `SUM(CASE WHEN x > n THEN 1 ELSE 0 END)` | igual que arriba |
| `SUM(a) FILTER (WHERE b = v)` | `apply(lambda x: x.loc[x['b']==v, 'a'].sum())` |
| `SUM(CASE WHEN x > n THEN a ELSE 0 END)` | `apply(lambda x: x.loc[x['x']>n, 'a'].sum())` |
| `CASE WHEN agg > v THEN 'High' ELSE 'Low' END` | `def` con `if/elif/else` dentro del `apply` |

**Next:** `07_merging_joining.ipynb` — combining Adobe + paid data into one DataFrame.